# Pixel-level bound tightness vs semantic regions



In [2]:
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
import matplotlib.pyplot as plt

from auto_LiRPA import BoundedModule, BoundedTensor, PerturbationLpNorm
import torchvision

# ----------------------------
# 0) Config
# ----------------------------
device = "cpu"          # change to "cuda" if you have GPU + enough VRAM
eps = 2/255.0           # Linf perturbation in [0,1] pixel space
K = 10                  # number of competitor classes to verify margins against (top-K)
img_path = "../benchmarks/vggnet16_benchmark2022/imagenet-sample/n01558993_robin.JPEG"

torch.set_grad_enabled(False)

# ----------------------------
# 1) Load image + preprocess (ImageNet VGG style)
# ----------------------------
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1).to(device)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1).to(device)

def preprocess_pil(pil_img):
    pil_img = pil_img.convert("RGB")
    pil_img = pil_img.resize((256, 256))
    left = (256 - 224)//2
    top  = (256 - 224)//2
    pil_img = pil_img.crop((left, top, left+224, top+224))

    x = torch.from_numpy(np.array(pil_img)).float().to(device) / 255.0  # [H,W,3] in [0,1]
    x = x.permute(2,0,1).unsqueeze(0)                                    # [1,3,224,224]
    x_norm = (x - IMAGENET_MEAN) / IMAGENET_STD
    return x, x_norm, pil_img

pil = Image.open(img_path)
x_01, x_norm, pil_crop = preprocess_pil(pil)

# ----------------------------
# 2) Input bounds + "initial bounding heatmap" (same 224x224)
# ----------------------------
# Optional mask: [1,1,224,224], 1=perturb allowed. Replace with your SAM mask if you want.
mask = torch.ones((1,1,224,224), dtype=torch.float32, device=device)

lb_01 = torch.clamp(x_01 - eps * mask, 0.0, 1.0)
ub_01 = torch.clamp(x_01 + eps * mask, 0.0, 1.0)

# Heatmap = average (ub-lb) across RGB channels -> [224,224]
heat = (ub_01 - lb_01).mean(dim=1).squeeze(0).detach().cpu().numpy()

# Convert bounds to normalized space (model input space)
lb_norm = (lb_01 - IMAGENET_MEAN) / IMAGENET_STD
ub_norm = (ub_01 - IMAGENET_MEAN) / IMAGENET_STD

# ----------------------------
# 3) Load VGG16 pretrained + fix dropout for auto_LiRPA
# ----------------------------
model = torchvision.models.vgg16(
    weights=torchvision.models.VGG16_Weights.IMAGENET1K_V1
).to(device)

# IMPORTANT: auto_LiRPA wants dropout parsable in train() mode
model.train()

# Make dropout deterministic / inactive
for m in model.modules():
    if isinstance(m, torch.nn.Dropout):
        m.p = 0.0

# Predict label on clean input
logits = model(x_norm)
y = int(logits.argmax(dim=1).item())
print("Predicted class id:", y)

# pick top-K competitors (excluding y)
topk = torch.topk(logits, k=K, dim=1).indices.squeeze(0).tolist()
competitors = [k for k in topk if k != y]
if len(competitors) == 0:
    competitors = [int(torch.topk(logits, k=K+1, dim=1).indices.squeeze(0)[-1].item())]

# ----------------------------
# 4) Margin model against selected competitors
# ----------------------------
class SelectedMarginModel(nn.Module):
    def __init__(self, base, true_y, competitors):
        super().__init__()
        self.base = base
        self.true_y = int(true_y)
        self.competitors = list(competitors)

    def forward(self, x):
        out = self.base(x)                          # [B,1000]
        yt = out[:, self.true_y:self.true_y+1]      # [B,1]
        yc = out[:, self.competitors]               # [B,K']
        return yt - yc                              # [B,K']

margin_model = SelectedMarginModel(model, y, competitors).to(device)

# ----------------------------
# 5) Wrap with auto_LiRPA and compute initial bounds (no BaB)
# ----------------------------
dummy = torch.zeros_like(x_norm)
bound_opts = {"conv_mode": "hybrid"}  # safer than "patches" (patches can OOM)
bm = BoundedModule(margin_model, dummy, bound_opts=bound_opts)

ptb = PerturbationLpNorm(norm=np.inf, x_L=lb_norm, x_U=ub_norm)
x_bounded = BoundedTensor(x_norm, ptb)

lb_m, ub_m = bm.compute_bounds(x=(x_bounded,), method="CROWN")
lb_m = lb_m.squeeze(0).detach().cpu().numpy()
ub_m = ub_m.squeeze(0).detach().cpu().numpy()

worst_idx = int(lb_m.argmin())
worst_comp = competitors[worst_idx]
worst_lb = float(lb_m[worst_idx])

print("Competitors (subset):", competitors)
print("Worst competitor id (subset):", worst_comp)
print("Worst certified margin LB (subset):", worst_lb)

# ----------------------------
# 6) Plot image + initial bound-width heatmap overlay
# ----------------------------
fig = plt.figure(figsize=(10,4))

ax1 = plt.subplot(1,2,1)
ax1.imshow(pil_crop)
ax1.set_title("Robin crop (224×224)")
ax1.axis("off")

ax2 = plt.subplot(1,2,2)
ax2.imshow(pil_crop)
ax2.imshow(heat, alpha=0.6)
ax2.set_title(f"Initial input bound width heatmap (eps={eps:.5f})")
ax2.axis("off")

plt.tight_layout()
plt.show()

Predicted class id: 15


/Users/zd3504phd/miniforge3/envs/xaiv/lib/python3.11/site-packages/torch/onnx/symbolic_helper.py:1515: UserWarning: ONNX export mode is set to TrainingMode.EVAL, but operator 'dropout' is set to train=True. Exporting with train=True.
  warnings.warn(


: 